In [1]:
from plotly.io import show
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

from skfolio import Population, RiskMeasure
from skfolio.cluster import HierarchicalClustering, LinkageMethod
from skfolio.datasets import load_sp500_dataset
from skfolio.distance import KendallDistance
from skfolio.optimization import (
    EqualWeighted,
    MeanRisk,
    NestedClustersOptimization,
    ObjectiveFunction,
    RiskBudgeting,
)
from skfolio.preprocessing import prices_to_returns

prices = load_sp500_dataset()

X = prices_to_returns(prices)
X_train, X_test = train_test_split(X, test_size=0.33, shuffle=False)

In [2]:
inner_estimator = MeanRisk(
    objective_function=ObjectiveFunction.MAXIMIZE_RATIO,
    risk_measure=RiskMeasure.VARIANCE,
)
outer_estimator = RiskBudgeting(risk_measure=RiskMeasure.CVAR)

model1 = NestedClustersOptimization(
    inner_estimator=inner_estimator,
    outer_estimator=outer_estimator,
    n_jobs=-1,
    portfolio_params=dict(name="NCO-1"),
)
model1.fit(X_train)
model1.weights_

array([4.34541668e-02, 3.10827859e-03, 2.62488372e-08, 4.55212307e-02,
       7.75889069e-02, 6.00676996e-02, 4.72964387e-02, 1.25134800e-01,
       4.62242005e-02, 3.16526010e-02, 5.09579924e-08, 2.40362184e-03,
       8.63999339e-02, 3.19344084e-02, 6.00539403e-02, 4.79349448e-02,
       9.30715227e-02, 5.98339728e-02, 4.48377475e-02, 9.34815076e-02])

In [3]:
model1.clustering_estimator_.plot_dendrogram(heatmap=False)

In [4]:
model1.clustering_estimator_.plot_dendrogram()

In [5]:
model2 = NestedClustersOptimization(
    inner_estimator=inner_estimator,
    outer_estimator=outer_estimator,
    clustering_estimator=HierarchicalClustering(
        linkage_method=LinkageMethod.SINGLE,
    ),
    n_jobs=-1,
    portfolio_params=dict(name="NCO-2"),
)
model2.fit(X_train)
model2.clustering_estimator_.plot_dendrogram(heatmap=True)

In [6]:
model3 = NestedClustersOptimization(
    inner_estimator=inner_estimator,
    outer_estimator=outer_estimator,
    distance_estimator=KendallDistance(absolute=True),
    n_jobs=-1,
    portfolio_params=dict(name="NCO-3"),
)
model3.fit(X_train)
model3.clustering_estimator_.plot_dendrogram(heatmap=True)

In [7]:
model4 = NestedClustersOptimization(
    inner_estimator=inner_estimator,
    outer_estimator=outer_estimator,
    clustering_estimator=KMeans(n_init="auto"),
    n_jobs=-1,
    portfolio_params=dict(name="NCO-4"),
)
model4.fit(X_train)
model4.weights_

array([7.38850960e-02, 1.80520545e-02, 2.37335143e-08, 2.92357133e-02,
       7.01047557e-02, 5.43116478e-02, 2.15276747e-02, 1.08402417e-01,
       4.17947169e-02, 5.47482847e-02, 4.41441512e-08, 2.08222187e-03,
       5.18104899e-02, 5.05362644e-02, 5.20238358e-02, 7.57146996e-02,
       8.33490351e-02, 1.05593959e-01, 2.23626946e-02, 8.44643714e-02])

In [8]:
bench = EqualWeighted()
bench.fit(X_train)
bench.weights_

array([0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05,
       0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05, 0.05])

In [9]:
population_test = Population([])
for model in [model1, model2, model3, model4, bench]:
    population_test.append(model.predict(X_test))

population_test.plot_cumulative_returns()

In [10]:
fig = population_test.plot_composition()
show(fig)